# English all-token pooling — SVM
Three experiments in one notebook; one cumulative CSV for SVM.

**Status:** Four-C search added; full C-grid runs have not been executed.
**Restart the kernel and Run All** so the new C list, evaluator, and output path take effect.
Run the definition/data cells once, then the three experiment cells in order (or independently after initialization).
Completed representations resume from this algorithm's CSV without duplicate baseline rows.

## Methods
- Original English dataset/model/cleaning: `34_self_contained_bert_mean_mean_cls3_lr_svm.ipynb`.
- Arabic reference: `experiment_svm_l2_all_tokens_greater_than_k_baselines.py`, the lower/upper-band runner, and `src/classifier_search.py` / `src/evaluation.py`.
- Fixed Arabic settings: seed 42, batch 32, maximum length **320**, 10 shuffled stratified training folds, fold-local StandardScaler for LR/SVM, no scaler for RF; only SVM C is varied: **[0.1, 0.01, 0.001, 1]**. LinearSVC (dual=False), class weights, and max_iter remain unchanged.
- English BERT replaces MARBERT, and the original English cleaning/stratified 80/20 split replaces the Arabic official split.
- No project imports, edits, installs, downloads, saved embeddings, or separate checkpoint files.
- Final-layer CLS and MEAN baselines; filtered methods are **mean only**, without CLS concatenation.

### Filtering convention
Token L2 norms are standardized **within each sentence**, using population standard deviation over **all attended tokens, including CLS/SEP and other special tokens**. Padding is excluded.
`mean_lt_k` keeps `abs(z) < k`; `mean_gt_k` keeps `abs(z) > k`; combined keeps `lower < abs(z) < upper`.
All 4 × 11 lower/upper combinations are tested. Unlike the old inclusive Arabic K-grid (`<=`), the requested LT operator is **strict**.
If nothing passes, fall back to the full attended-token mean, matching the Arabic band/GT runner, and report the fallback count.
Special tokens are eligible, not forced to survive the filter.

### Evaluation
Each training fold's threshold is optimized on that fold's validation labels, exactly as in the Arabic experiments; these threshold-tuned CV scores are **not nested/unbiased estimates**.
For each representation, all four C values use the same 10 folds. Select C by mean validation Macro-F1 (ties: accuracy, macro recall, then original C order). The selected C is refitted on all training data.
The final threshold comes from the selected C's pooled training OOF scores; test labels are used only for reporting. Test is evaluated once per representation, after C selection.
Do not select K using held-out test scores.

### Output
`results/english_all_tokens_svm_c_grid_results.csv` contains 61 CV-selected method rows, the winning C, test metrics, all four C summaries and all 40 fold records as JSON, the selected C's 10 fold records, parameters, data/setup provenance, token counts, and fallback counts.
The existing fixed-C file `english_all_tokens_svm_results.csv` is preserved and is not used to skip this new search.
The file is updated after **each representation**, and is complete after all three experiment cells. Run only one instance of this algorithm notebook at a time.


## 1. Imports and SVM C-search settings
Use your existing GPU environment; this cell installs nothing. Only C varies; BERT and all other settings remain fixed.


In [1]:
from __future__ import annotations

import gc
import hashlib
import html
import io
import json
import os
import random
import re
import tempfile
import time
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import sklearn
import transformers
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from transformers import AutoModel, AutoTokenizer

# Arabic all-token settings remain fixed; only SVM C is searched.
SEED = 42
MAX_LENGTH = 320
BATCH_SIZE = 32
CV_FOLDS = 10
TEST_SIZE = 0.20
MODEL_NAME = "bert-base-uncased"
LT_K_VALUES = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5]
GT_K_VALUES = [0.0001, 0.001, 0.01, 0.1]
CSV_NAME = "English Sarcasm.csv"
ORIGINAL_ZIP = Path(os.environ.get("MFMP_ENGLISH_ZIP", "data/raw/english/archive.zip"))
EXTRACTED_CSV = Path(os.environ.get("MFMP_ENGLISH_CSV", "data/raw/english/English Sarcasm.csv"))

# Works when launched either from the project root or from notebooks/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "experiments").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from this repository's root or experiments folder.")

CLASSIFIER = "SVM"
C_VALUES = [0.1, 0.01, 0.001, 1]
PARAMETERS = {"class_weight": "balanced", "max_iter": 10000}
RESULTS_PATH = PROJECT_ROOT / "results" / "english_all_tokens_svm_c_grid_results.csv"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Algorithm:", CLASSIFIER, "| fixed parameters:", PARAMETERS, "| C values:", C_VALUES)
print("Model:", MODEL_NAME, "| max length:", MAX_LENGTH, "| batch:", BATCH_SIZE)
print("Device:", DEVICE, "| CV folds:", CV_FOLDS, "| seed:", SEED)
print("One cumulative results file:", RESULTS_PATH)


<PYTHON_ENV>\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Algorithm: SVM | fixed parameters: {'class_weight': 'balanced', 'max_iter': 10000} | C values: [0.1, 0.01, 0.001, 1]
Model: bert-base-uncased | max length: 320 | batch: 32
Device: cuda | CV folds: 10 | seed: 42
One cumulative results file: <PROJECT_ROOT>\results\english_all_tokens_svm_c_grid_results.csv


## 2. Load and prepare the English dataset
Reads the original ZIP if present, otherwise the verified extracted English CSV. Does not change either source.


In [2]:
# Read only the known English dataset; never use the Arabic CSVs.
if ORIGINAL_ZIP.is_file():
    with zipfile.ZipFile(ORIGINAL_ZIP) as archive:
        csv_bytes = archive.read(CSV_NAME)
    DATA_SOURCE = f"{ORIGINAL_ZIP}::{CSV_NAME}"
elif EXTRACTED_CSV.is_file():
    csv_bytes = EXTRACTED_CSV.read_bytes()
    DATA_SOURCE = str(EXTRACTED_CSV)
else:
    raise FileNotFoundError(
        f"English dataset not found. Restore {ORIGINAL_ZIP} (containing {CSV_NAME}) "
        f"or {EXTRACTED_CSV}. No data was downloaded or substituted."
    )
DATA_SHA256 = hashlib.sha256(csv_bytes).hexdigest()
raw_df = pd.read_csv(io.BytesIO(csv_bytes))
if not {"Tweet", "Label"}.issubset(raw_df.columns):
    raise ValueError("The English CSV must have Tweet and Label columns.")

# The English cleaning and deduplication below are copied from notebook 34.
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")

def clean_tweet(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text

df = raw_df[["Tweet", "Label"]].copy()
df["clean_text"] = df["Tweet"].map(clean_tweet)
df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
df = df.dropna(subset=["clean_text", "Label"])
df = df[df["clean_text"].str.len() > 0]
df = df[df["Label"].isin([0, 1])]
df["Label"] = df["Label"].astype(int)
df = df.drop_duplicates(subset=["clean_text", "Label"]).reset_index(drop=True)

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["Label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df["Label"].to_numpy()
y_test = test_df["Label"].to_numpy()
if set(np.unique(y_train)) != {0, 1}:
    raise ValueError("Both binary classes 0 and 1 are required.")
if np.bincount(y_train).min() < CV_FOLDS:
    raise ValueError("There are too few training examples per class for 10-fold CV.")

# Every representation and algorithm uses the same training folds.
FOLD_INDICES = list(StratifiedKFold(
    n_splits=CV_FOLDS, shuffle=True, random_state=SEED,
).split(np.zeros(len(y_train)), y_train))
print("Data source:", DATA_SOURCE)
print("Raw / cleaned rows:", len(raw_df), "/", len(df))
print("Train / test rows:", len(train_df), "/", len(test_df))
display(pd.DataFrame({
    "train": train_df["Label"].value_counts().sort_index(),
    "test": test_df["Label"].value_counts().sort_index(),
}))


Data source: data/raw/english/English Sarcasm.csv
Raw / cleaned rows: 4553 / 4546
Train / test rows: 3636 / 910


,train,test
Label,,
0,2836,710
1,800,200


## 3. Representation rules and all-token extraction
61 rules, sentence-local L2 z-scores, strict bounds, and counted empty-filter fallbacks.


In [3]:
def rule(name, kind, lower_k=None, upper_k=None):
    return {"method": name, "kind": kind, "lower_k": lower_k, "upper_k": upper_k}


BASELINE_RULES = [rule("cls", "cls"), rule("mean", "mean")]
LT_RULES = [rule(f"mean_lt_{k:g}", "mean_lt_k", upper_k=k) for k in LT_K_VALUES]
GT_RULES = [rule(f"mean_gt_{k:g}", "mean_gt_k", lower_k=k) for k in GT_K_VALUES]
BAND_RULES = [
    rule(f"mean_gt_{lower:g}_lt_{upper:g}", "mean_gt_k_lt_k", lower, upper)
    for lower in GT_K_VALUES for upper in LT_K_VALUES
]
ALL_RULES = BASELINE_RULES + LT_RULES + GT_RULES + BAND_RULES
RULE_INDEX = {r["method"]: r for r in ALL_RULES}
assert len(RULE_INDEX) == len(ALL_RULES) == 61
assert len(LT_RULES) == 11 and len(GT_RULES) == 4 and len(BAND_RULES) == 44


# Same sentence-local population standard deviation and mean pooling as Arabic.
def l2_z_scores(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    valid_float = attention_mask.bool().float()
    counts = torch.clamp(valid_float.sum(dim=1, keepdim=True), min=1.0)
    token_l2 = torch.linalg.vector_norm(hidden_states, ord=2, dim=-1)
    mean_l2 = torch.sum(token_l2 * valid_float, dim=1, keepdim=True) / counts
    centered_l2 = (token_l2 - mean_l2) * valid_float
    variance_l2 = torch.sum(centered_l2 * centered_l2, dim=1, keepdim=True) / counts
    std_l2 = torch.sqrt(torch.clamp(variance_l2, min=0.0))
    return (token_l2 - mean_l2) / torch.clamp(std_l2, min=1e-12)


def mean_pool(hidden_states: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    expanded_mask = mask.unsqueeze(-1).expand(hidden_states.size()).float()
    counts = torch.clamp(expanded_mask.sum(dim=1), min=1e-9)
    return torch.sum(hidden_states * expanded_mask, dim=1) / counts


def mask_for_rule(abs_z, valid_mask, item):
    if item["kind"] in {"cls", "mean"}:
        return valid_mask, 0
    keep = valid_mask.clone()
    if item["lower_k"] is not None:
        keep &= abs_z > item["lower_k"]
    if item["upper_k"] is not None:
        keep &= abs_z < item["upper_k"]
    has_tokens = keep.any(dim=1, keepdim=True)
    fallback_count = int((~has_tokens).sum().item())
    # Same empty-filter fallback as the Arabic all-token band/greater-than runs.
    return torch.where(has_tokens, keep, valid_mask), fallback_count


tokenizer = None
bert_model = None


def load_english_model():
    global tokenizer, bert_model
    if bert_model is None:
        # Reuse the existing BERT cache. No model downloads or setup changes.
        new_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
        new_model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True).to(DEVICE)
        new_model.eval()
        tokenizer, bert_model = new_tokenizer, new_model
    return tokenizer, bert_model


def extract_representations(texts, items, split_name):
    tok, model = load_english_model()
    texts = list(texts)
    # Final vectors only, not a cache of every token's hidden states.
    vectors = {
        item["method"]: np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
        for item in items
    }
    stats = {
        item["method"]: dict(total_valid_tokens=0, total_special_tokens=0,
                            included_tokens=0, included_special_tokens=0,
                            excluded_tokens=0, excluded_special_tokens=0,
                            filter_fallback_texts=0)
        for item in items
    }
    special_ids = set(tok.all_special_ids)
    print(f"Extracting {split_name}: {len(texts)} texts, {len(items)} representations")
    with torch.inference_mode():
        for start in range(0, len(texts), BATCH_SIZE):
            encoded = tok(
                texts[start:start + BATCH_SIZE], padding=True, truncation=True,
                max_length=MAX_LENGTH, return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            hidden = model(**encoded).last_hidden_state
            valid = encoded["attention_mask"].bool()
            # Special tokens participate in the statistics and filtering.
            # Only positions with attention_mask=0 (padding) are excluded.
            abs_z = l2_z_scores(hidden, encoded["attention_mask"]).abs()
            special = torch.zeros_like(valid)
            for special_id in special_ids:
                special |= encoded["input_ids"] == special_id
            special &= valid
            total_valid = int(valid.sum().item())
            total_special = int(special.sum().item())
            stop = start + hidden.shape[0]
            for item in items:
                keep, fallback = mask_for_rule(abs_z, valid, item)
                if item["kind"] == "cls":
                    pooled = hidden[:, 0, :]
                    # Counts reflect the single vector actually used by CLS.
                    keep = torch.zeros_like(valid)
                    keep[:, 0] = valid[:, 0]
                else:
                    pooled = mean_pool(hidden, keep)
                vectors[item["method"]][start:stop] = pooled.cpu().numpy().astype(np.float32)
                included = int(keep.sum().item())
                included_special = int((keep & special).sum().item())
                counts = stats[item["method"]]
                counts["total_valid_tokens"] += total_valid
                counts["total_special_tokens"] += total_special
                counts["included_tokens"] += included
                counts["included_special_tokens"] += included_special
                counts["excluded_tokens"] += total_valid - included
                counts["excluded_special_tokens"] += total_special - included_special
                counts["filter_fallback_texts"] += fallback
            if stop == len(texts) or (start // BATCH_SIZE + 1) % 20 == 0:
                print(f"  {split_name}: {stop}/{len(texts)}")
    for values in vectors.values():
        if not np.isfinite(values).all():
            raise ValueError("Non-finite embedding values; no results were saved.")
    return vectors, stats


print("Representations: 2 baselines + 11 LT + 4 GT + 44 bands = 61")


Representations: 2 baselines + 11 LT + 4 GT + 44 bands = 61


## 4. SVM C search with the Arabic CV/threshold procedure
Compare C = [0.1, 0.01, 0.001, 1] on the same 10 training folds; select C using validation scores only. Metric and threshold functions remain unchanged.


In [4]:
def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: list[int] | None = None,
) -> dict[str, Any]:
    labels = labels or [0, 1]
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    per_class_precision, per_class_recall, per_class_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "precision_class_0": per_class_precision[0],
        "recall_class_0": per_class_recall[0],
        "f1_class_0": per_class_f1[0],
        "precision_class_1": per_class_precision[1],
        "recall_class_1": per_class_recall[1],
        "f1_class_1": per_class_f1[1],
    }


def select_best_threshold(
    y_true: np.ndarray,
    scores: np.ndarray,
    verbose: bool = True,
) -> tuple[float, dict[str, Any]]:
    """Select threshold by macro F1, accuracy, macro recall, then class-1 recall."""
    y_true = np.asarray(y_true).ravel()
    scores = np.asarray(scores).ravel()
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must contain the same number of samples.")
    if len(y_true) == 0:
        raise ValueError("Threshold selection requires at least one sample.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("Threshold selection requires binary labels encoded as 0 and 1.")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = y_true[order]
    unique_scores, first_indices = np.unique(sorted_scores, return_index=True)

    total_positive = int(np.sum(y_true == 1))
    total_negative = int(np.sum(y_true == 0))
    tp = total_positive
    fp = total_negative
    fn = 0
    tn = 0

    def metric_row(threshold: float) -> dict[str, Any]:
        precision_0 = tn / (tn + fn) if tn + fn else 0.0
        recall_0 = tn / (tn + fp) if tn + fp else 0.0
        f1_0 = (
            2 * precision_0 * recall_0 / (precision_0 + recall_0)
            if precision_0 + recall_0
            else 0.0
        )
        precision_1 = tp / (tp + fp) if tp + fp else 0.0
        recall_1 = tp / (tp + fn) if tp + fn else 0.0
        f1_1 = (
            2 * precision_1 * recall_1 / (precision_1 + recall_1)
            if precision_1 + recall_1
            else 0.0
        )
        present_class_0 = total_negative > 0 or tn + fn > 0
        present_class_1 = total_positive > 0 or tp + fp > 0
        present_count = int(present_class_0) + int(present_class_1)
        macro_precision = (
            precision_0 * present_class_0 + precision_1 * present_class_1
        ) / present_count
        macro_recall = (
            recall_0 * present_class_0 + recall_1 * present_class_1
        ) / present_count
        macro_f1 = (f1_0 * present_class_0 + f1_1 * present_class_1) / present_count
        return {
            "threshold": float(threshold),
            "accuracy": (tn + tp) / len(y_true),
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "precision_class_0": precision_0,
            "recall_class_0": recall_0,
            "f1_class_0": f1_0,
            "precision_class_1": precision_1,
            "recall_class_1": recall_1,
            "f1_class_1": f1_1,
        }

    best = metric_row(unique_scores[0])
    best_key = (
        best["macro_f1"],
        best["accuracy"],
        best["macro_recall"],
        best["recall_class_1"],
    )

    for unique_index in range(len(unique_scores) - 1):
        start = first_indices[unique_index]
        end = first_indices[unique_index + 1]
        removed_labels = sorted_labels[start:end]
        removed_positive = int(np.sum(removed_labels == 1))
        removed_negative = int(np.sum(removed_labels == 0))
        tp -= removed_positive
        fn += removed_positive
        fp -= removed_negative
        tn += removed_negative

        threshold = (unique_scores[unique_index] + unique_scores[unique_index + 1]) / 2.0
        row = metric_row(threshold)
        row_key = (
            row["macro_f1"],
            row["accuracy"],
            row["macro_recall"],
            row["recall_class_1"],
        )
        if row_key > best_key:
            best = row
            best_key = row_key

    if verbose:
        print("Best validation threshold:")
        print(json.dumps(best, indent=2))
    return float(best["threshold"]), best


def build_estimator(parameters):
    # Arabic estimator/scaler choices; parameters contain the current candidate C.
    if CLASSIFIER == "LR":
        classifier = LogisticRegression(random_state=SEED, **parameters)
        return Pipeline([("scaler", StandardScaler()), ("classifier", classifier)])
    if CLASSIFIER == "SVM":
        classifier = LinearSVC(dual=False, random_state=SEED, **parameters)
        return Pipeline([("scaler", StandardScaler()), ("classifier", classifier)])
    if CLASSIFIER == "RF":
        return RandomForestClassifier(random_state=SEED, n_jobs=-1, **parameters)
    raise ValueError(CLASSIFIER)


def positive_scores(estimator, values):
    if hasattr(estimator, "predict_proba"):
        positive_index = int(np.flatnonzero(estimator.classes_ == 1)[0])
        return estimator.predict_proba(values)[:, positive_index], "probability_class_1"
    return np.asarray(estimator.decision_function(values)).ravel(), "decision_score"


def score_predictions(labels, predictions):
    return {
        **calculate_metrics(labels, predictions),
        "tn": int(np.sum((labels == 0) & (predictions == 0))),
        "fp": int(np.sum((labels == 0) & (predictions == 1))),
        "fn": int(np.sum((labels == 1) & (predictions == 0))),
        "tp": int(np.sum((labels == 1) & (predictions == 1))),
        "correct": int(np.sum(labels == predictions)),
        "incorrect": int(np.sum(labels != predictions)),
    }


def evaluate_representation(item, x_train, x_test):
    started = time.perf_counter()
    candidates = []
    all_fold_rows = []
    # Reuse the same embeddings and same 10 folds for every candidate C.
    for candidate_index, c_value in enumerate(C_VALUES, 1):
        parameters = {**PARAMETERS, "C": c_value}
        base = build_estimator(parameters)
        oof_scores = np.empty(len(y_train), dtype=float)
        fold_rows = []
        candidate_started = time.perf_counter()
        for fold_number, (fit_index, val_index) in enumerate(FOLD_INDICES, 1):
            estimator = clone(base)
            estimator.fit(x_train[fit_index], y_train[fit_index])
            scores, score_type = positive_scores(estimator, x_train[val_index])
            oof_scores[val_index] = scores
            threshold, _ = select_best_threshold(y_train[val_index], scores, verbose=False)
            metrics = score_predictions(y_train[val_index], (scores >= threshold).astype(int))
            fold_rows.append({
                "parameter_set": candidate_index, "param_C": c_value,
                "fold": fold_number, "train_size": len(fit_index),
                "validation_size": len(val_index), "best_threshold": threshold, **metrics,
            })
            print(f"  {item['method']} C={c_value:g} fold {fold_number}/{CV_FOLDS}: "
                  f"macro_f1={metrics['macro_f1']:.6f}")

        threshold, _ = select_best_threshold(y_train, oof_scores, verbose=False)
        oof_metrics = score_predictions(y_train, (oof_scores >= threshold).astype(int))
        folds = pd.DataFrame(fold_rows)
        summary = {
            "parameter_set": candidate_index, "param_C": c_value,
            "parameters": json.dumps(parameters, sort_keys=True),
            "cv_macro_f1_mean": float(folds["macro_f1"].mean()),
            "cv_macro_f1_std": float(folds["macro_f1"].std(ddof=1)),
            "cv_accuracy_mean": float(folds["accuracy"].mean()),
            "cv_accuracy_std": float(folds["accuracy"].std(ddof=1)),
            "cv_macro_recall_mean": float(folds["macro_recall"].mean()),
            "fold_threshold_mean": float(folds["best_threshold"].mean()),
            "fold_threshold_std": float(folds["best_threshold"].std(ddof=1)),
            "best_threshold": threshold,
            **{f"oof_{key}": value for key, value in oof_metrics.items()},
            "candidate_fit_seconds": time.perf_counter() - candidate_started,
        }
        candidates.append(summary)
        all_fold_rows.extend(fold_rows)

    # Same CV selection order as the Arabic classifier search. TEST is not used.
    best = max(candidates, key=lambda row: (
        row["cv_macro_f1_mean"], row["cv_accuracy_mean"], row["cv_macro_recall_mean"],
    ))
    best_parameters = json.loads(best["parameters"])
    threshold = best["best_threshold"]
    final_estimator = build_estimator(best_parameters)
    final_estimator.fit(x_train, y_train)
    test_scores, score_type = positive_scores(final_estimator, x_test)
    test_metrics = score_predictions(y_test, (test_scores >= threshold).astype(int))
    print(f"Selected {item['method']}: C={best['param_C']:g}, "
          f"CV macro-F1={best['cv_macro_f1_mean']:.6f}")
    return {
        **item, "classifier": CLASSIFIER, **best,
        **{f"param_{key}": value for key, value in best_parameters.items()},
        "c_values_json": json.dumps(C_VALUES),
        "parameter_sets_tested": len(candidates),
        "c_selection_source": "mean_10_fold_validation_macro_f1",
        "threshold_source": "pooled_out_of_fold_train_scores",
        "score_type": score_type,
        **test_metrics,
        # Keep all four C summaries and all 40 folds in the same algorithm CSV.
        "c_search_results_json": json.dumps(candidates, sort_keys=True),
        "cv_fold_results_json": json.dumps(all_fold_rows, sort_keys=True),
        "selected_cv_fold_results_json": json.dumps([
            row for row in all_fold_rows if row["parameter_set"] == best["parameter_set"]
        ], sort_keys=True),
        "fit_seconds": time.perf_counter() - started,
    }


## 5. Cumulative result saving and experiment runner
Each successful method is saved atomically to the single algorithm CSV. A different data/setup fingerprint is refused rather than overwriting previous results.


In [5]:
# Fingerprint prevents mixing another dataset/setup into a previous result file.
RUN_CONTEXT = {
    "implementation": "english-all-token-svm-c-grid-v2",
    "data_sha256": DATA_SHA256,
    "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
    "seed": SEED, "test_size": TEST_SIZE, "cv_folds": CV_FOLDS,
    "cv_shuffle": True, "classifier": CLASSIFIER, "parameters": PARAMETERS,
    "c_values": C_VALUES,
    "c_selection": "CV macro-F1, accuracy, macro recall; first-listed C breaks exact ties",
    "lt_k_values": LT_K_VALUES, "gt_k_values": GT_K_VALUES,
    "filter": "strict lower < abs(sentence-local population L2 z-score) < upper",
    "token_population": "all attended tokens including special tokens; padding excluded",
    "fallback": "all attended tokens if filtered mask is empty",
    "cleaning": "notebook_34_english_clean_tweet_and_dedup",
    "train_rows": len(y_train), "test_rows": len(y_test),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "transformers": transformers.__version__},
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(RUN_CONTEXT, sort_keys=True).encode("utf-8")
).hexdigest()


def load_saved_results():
    if not RESULTS_PATH.exists():
        return pd.DataFrame()
    saved = pd.read_csv(RESULTS_PATH)
    required = {"method", "classifier", "run_signature", "macro_f1", "cv_macro_f1_mean"}
    if not required.issubset(saved.columns) or saved.empty:
        raise ValueError(f"Not a valid completed English results file: {RESULTS_PATH}")
    if not saved["run_signature"].eq(RUN_SIGNATURE).all():
        raise ValueError(
            f"{RESULTS_PATH} belongs to a different setup/dataset. "
            "It will not be overwritten. Use a new output filename for a different run."
        )
    if not saved["classifier"].eq(CLASSIFIER).all() or saved["method"].duplicated().any():
        raise ValueError("Result file has mixed algorithms or duplicate methods.")
    if not set(saved["method"]).issubset(RULE_INDEX):
        raise ValueError("Result file contains unexpected representation names.")
    return saved


def save_result(row):
    saved = load_saved_results()
    # Resume is intentional: a completed method is not retrained or duplicated.
    if not saved.empty and row["method"] in set(saved["method"]):
        return saved
    combined = pd.DataFrame(saved.to_dict("records") + [row])
    order = {item["method"]: index for index, item in enumerate(ALL_RULES)}
    combined = combined.sort_values("method", key=lambda s: s.map(order)).reset_index(drop=True)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        # Write in the destination directory so replacement is atomic.
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".tmp", prefix=RESULTS_PATH.stem + "_",
            dir=RESULTS_PATH.parent, encoding="utf-8-sig", newline="", delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            combined.to_csv(handle, index=False)
        os.replace(temporary_path, RESULTS_PATH)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()
    print(f"Saved {row['method']} -> {RESULTS_PATH.name} ({len(combined)}/61)")
    return combined


def show_results(frame):
    columns = [
        "method", "param_C", "lower_k", "upper_k", "cv_macro_f1_mean", "cv_macro_f1_std",
        "accuracy", "macro_precision", "macro_recall", "macro_f1", "f1_class_1",
        "tn", "fp", "fn", "tp", "test_included_tokens", "test_included_special_tokens",
        "test_filter_fallback_texts",
    ]
    display(frame[[column for column in columns if column in frame]].reset_index(drop=True))


def run_experiment(experiment_name, items):
    requested = BASELINE_RULES + items
    saved = load_saved_results()
    completed = set(saved["method"]) if not saved.empty else set()
    pending = [item for item in requested if item["method"] not in completed]
    if pending:
        train_vectors, train_stats = extract_representations(
            train_df["clean_text"], pending, "train",
        )
        test_vectors, test_stats = extract_representations(
            test_df["clean_text"], pending, "test",
        )
        for item in pending:
            name = item["method"]
            row = evaluate_representation(item, train_vectors[name], test_vectors[name])
            row.update({
                "first_saved_in_experiment": experiment_name,
                "run_signature": RUN_SIGNATURE,
                "run_context_json": json.dumps(RUN_CONTEXT, sort_keys=True),
                "data_source": DATA_SOURCE, "data_sha256": DATA_SHA256,
                "model_name": MODEL_NAME, "max_length": MAX_LENGTH,
                "batch_size": BATCH_SIZE, "seed": SEED, "cv_folds": CV_FOLDS,
                "train_rows": len(y_train), "test_rows": len(y_test),
                "feature_dimensions": train_vectors[name].shape[1],
                "filter_operators": "strict < and >",
                "special_tokens_included": True,
                "padding_included": False,
                **{f"train_{key}": value for key, value in train_stats[name].items()},
                **{f"test_{key}": value for key, value in test_stats[name].items()},
            })
            save_result(row)
            del train_vectors[name], test_vectors[name]
        del train_vectors, test_vectors
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = load_saved_results()
    names = {item["method"] for item in requested}
    result = saved[saved["method"].isin(names)]
    print(f"{experiment_name}: {len(result)}/{len(requested)} complete")
    show_results(result)
    return result


print("Ready. Run each of the three experiment cells below.")
print("Baselines are saved once and reused in all three comparisons.")


Ready. Run each of the three experiment cells below.
Baselines are saved once and reused in all three comparisons.


## Experiment 1 — Less-than only
CLS + MEAN + 11 `mean_lt_k` settings (13 comparisons).


In [6]:
less_than_results = run_experiment("less_than", LT_RULES)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 489.30it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting train: 3636 texts, 13 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 13 representations
  test: 640/910
  test: 910/910
  cls C=0.1 fold 1/10: macro_f1=0.764361
  cls C=0.1 fold 2/10: macro_f1=0.833774
  cls C=0.1 fold 3/10: macro_f1=0.784870
  cls C=0.1 fold 4/10: macro_f1=0.818208
  cls C=0.1 fold 5/10: macro_f1=0.800730
  cls C=0.1 fold 6/10: macro_f1=0.795364
  cls C=0.1 fold 7/10: macro_f1=0.853036
  cls C=0.1 fold 8/10: macro_f1=0.831840
  cls C=0.1 fold 9/10: macro_f1=0.804839
  cls C=0.1 fold 10/10: macro_f1=0.843407
  cls C=0.01 fold 1/10: macro_f1=0.812591
  cls C=0.01 fold 2/10: macro_f1=0.851271
  cls C=0.01 fold 3/10: macro_f1=0.814846
  cls C=0.01 fold 4/10: macro_f1=0.835497
  cls C=0.01 fold 5/10: macro_f1=0.821901
  cls C=0.01 fold 6/10: macro_f1=0.822163
  cls C=0.01 fold 7/10: macro_f1=0.853643
  cls C=0.01 fold 8/10: macro_f1=0.847059
  cls C=0.01

,method,param_C,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,0.001,NaN,NaN,0.848997,0.012791,0.897802,0.860454,0.832148,0.845039,0.754617,674,36,57,143,910,910,0
1,mean,0.001,NaN,NaN,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15911,1820,0
2,mean_lt_0.5,0.001,NaN,0.5,0.862045,0.020454,0.887912,0.834841,0.841972,0.838316,0.748768,656,54,48,152,5709,866,2
3,mean_lt_1,0.001,NaN,1.0,0.887673,0.010071,0.893407,0.844938,0.843697,0.844315,0.756892,662,48,49,151,10328,1417,0
4,mean_lt_1.5,0.001,NaN,1.5,0.892535,0.013750,0.902198,0.865188,0.842148,0.852845,0.767624,674,36,53,147,13706,1725,0
5,mean_lt_2,0.001,NaN,2.0,0.893890,0.014733,0.896703,0.848702,0.851197,0.849939,0.766169,662,48,46,154,15401,1807,0
6,mean_lt_2.5,0.001,NaN,2.5,0.893196,0.011298,0.906593,0.870923,0.850352,0.859994,0.779221,675,35,50,150,15829,1817,0
7,mean_lt_3,0.001,NaN,3.0,0.894148,0.013290,0.906593,0.875594,0.843169,0.857815,0.774536,679,31,54,146,15901,1820,0
8,mean_lt_3.5,0.001,NaN,3.5,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15910,1820,0
9,mean_lt_4,0.001,NaN,4.0,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15911,1820,0


## Experiment 2 — Greater-than only
CLS + MEAN + 4 `mean_gt_k` settings (6 comparisons). Baselines are reused from the same file.


In [7]:
greater_than_results = run_experiment("greater_than", GT_RULES)


Extracting train: 3636 texts, 4 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 4 representations
  test: 640/910
  test: 910/910
  mean_gt_0.0001 C=0.1 fold 1/10: macro_f1=0.865881
  mean_gt_0.0001 C=0.1 fold 2/10: macro_f1=0.871831
  mean_gt_0.0001 C=0.1 fold 3/10: macro_f1=0.858644
  mean_gt_0.0001 C=0.1 fold 4/10: macro_f1=0.872461
  mean_gt_0.0001 C=0.1 fold 5/10: macro_f1=0.830247
  mean_gt_0.0001 C=0.1 fold 6/10: macro_f1=0.882445
  mean_gt_0.0001 C=0.1 fold 7/10: macro_f1=0.858541
  mean_gt_0.0001 C=0.1 fold 8/10: macro_f1=0.872855
  mean_gt_0.0001 C=0.1 fold 9/10: macro_f1=0.837366
  mean_gt_0.0001 C=0.1 fold 10/10: macro_f1=0.888204
  mean_gt_0.0001 C=0.01 fold 1/10: macro_f1=0.882249
  mean_gt_0.0001 C=0.01 fold 2/10: macro_f1=0.889343
  mean_gt_0.0001 C=0.01 fold 3/10: macro_f1=0.890842
  mean_gt_0.0001 C=0.01 fold 4/10: macro_f1=0.904837
  mean_gt_0.0001 C=0.01 fold

,method,param_C,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,0.001,NaN,NaN,0.848997,0.012791,0.897802,0.860454,0.832148,0.845039,0.754617,674,36,57,143,910,910,0
1,mean,0.001,NaN,NaN,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15911,1820,0
2,mean_gt_0.0001,0.001,0.0001,NaN,0.894553,0.013363,0.896703,0.846777,0.856585,0.851518,0.769608,659,51,43,157,15909,1820,0
3,mean_gt_0.001,0.001,0.0010,NaN,0.893662,0.013041,0.907692,0.876719,0.845669,0.859763,0.777778,679,31,53,147,15903,1820,0
4,mean_gt_0.01,0.001,0.0100,NaN,0.892709,0.012890,0.908791,0.877840,0.848169,0.861702,0.781003,679,31,52,148,15794,1800,0
5,mean_gt_0.1,0.001,0.1000,NaN,0.891819,0.012795,0.901099,0.859107,0.848627,0.853691,0.770408,669,41,49,151,14724,1631,0


## Experiment 3 — Greater-than AND less-than
CLS + MEAN + all 44 `mean_gt_lower_lt_upper` settings (46 comparisons). Baselines are reused.


In [8]:
combined_results = run_experiment("greater_and_less_than", BAND_RULES)


Extracting train: 3636 texts, 44 representations
  train: 640/3636
  train: 1280/3636
  train: 1920/3636
  train: 2560/3636
  train: 3200/3636
  train: 3636/3636
Extracting test: 910 texts, 44 representations
  test: 640/910
  test: 910/910
  mean_gt_0.0001_lt_0.5 C=0.1 fold 1/10: macro_f1=0.785812
  mean_gt_0.0001_lt_0.5 C=0.1 fold 2/10: macro_f1=0.798784
  mean_gt_0.0001_lt_0.5 C=0.1 fold 3/10: macro_f1=0.856556
  mean_gt_0.0001_lt_0.5 C=0.1 fold 4/10: macro_f1=0.848474
  mean_gt_0.0001_lt_0.5 C=0.1 fold 5/10: macro_f1=0.794080
  mean_gt_0.0001_lt_0.5 C=0.1 fold 6/10: macro_f1=0.807746
  mean_gt_0.0001_lt_0.5 C=0.1 fold 7/10: macro_f1=0.841526
  mean_gt_0.0001_lt_0.5 C=0.1 fold 8/10: macro_f1=0.799549
  mean_gt_0.0001_lt_0.5 C=0.1 fold 9/10: macro_f1=0.830189
  mean_gt_0.0001_lt_0.5 C=0.1 fold 10/10: macro_f1=0.825138
  mean_gt_0.0001_lt_0.5 C=0.01 fold 1/10: macro_f1=0.817283
  mean_gt_0.0001_lt_0.5 C=0.01 fold 2/10: macro_f1=0.833774
  mean_gt_0.0001_lt_0.5 C=0.01 fold 3/10: macro_

,method,param_C,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,0.001,NaN,NaN,0.848997,0.012791,0.897802,0.860454,0.832148,0.845039,0.754617,674,36,57,143,910,910,0
1,mean,0.001,NaN,NaN,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15911,1820,0
2,mean_gt_0.0001_lt_0.5,0.001,0.0001,0.5,0.862045,0.020454,0.889011,0.836689,0.842676,0.839620,0.750617,657,53,48,152,5707,866,2
3,mean_gt_0.0001_lt_1,0.001,0.0001,1.0,0.887565,0.009963,0.893407,0.844938,0.843697,0.844315,0.756892,662,48,49,151,10326,1417,0
4,mean_gt_0.0001_lt_1.5,0.001,0.0001,1.5,0.892633,0.013786,0.902198,0.865188,0.842148,0.852845,0.767624,674,36,53,147,13704,1725,0
5,mean_gt_0.0001_lt_2,0.001,0.0001,2.0,0.893890,0.014733,0.896703,0.848702,0.851197,0.849939,0.766169,662,48,46,154,15399,1807,0
6,mean_gt_0.0001_lt_2.5,0.001,0.0001,2.5,0.893196,0.011298,0.906593,0.870923,0.850352,0.859994,0.779221,675,35,50,150,15827,1817,0
7,mean_gt_0.0001_lt_3,0.001,0.0001,3.0,0.894148,0.013290,0.906593,0.875594,0.843169,0.857815,0.774536,679,31,54,146,15899,1820,0
8,mean_gt_0.0001_lt_3.5,0.001,0.0001,3.5,0.894553,0.013363,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15908,1820,0
9,mean_gt_0.0001_lt_4,0.001,0.0001,4.0,0.894553,0.013363,0.896703,0.846777,0.856585,0.851518,0.769608,659,51,43,157,15909,1820,0


## All options — one algorithm file
Displays all saved methods in the requested baseline/LT/GT/band order. Check that all 61 are present before interpreting the complete experiment.

The displayed `param_C` is the CV-selected C for each representation; all four C summaries are retained in `c_search_results_json`.


In [9]:
all_results = load_saved_results()
completed_methods = set(all_results["method"]) if not all_results.empty else set()
missing_methods = [item["method"] for item in ALL_RULES if item["method"] not in completed_methods]
print("Saved results:", RESULTS_PATH)
print(f"Completed: {len(completed_methods)}/61")
if missing_methods:
    print("Incomplete — still missing:", missing_methods)
else:
    print("All three experiments are complete.")
show_results(all_results)


Saved results: <PROJECT_ROOT>\results\english_all_tokens_svm_c_grid_results.csv
Completed: 61/61
All three experiments are complete.


,method,param_C,lower_k,upper_k,cv_macro_f1_mean,cv_macro_f1_std,accuracy,macro_precision,macro_recall,macro_f1,f1_class_1,tn,fp,fn,tp,test_included_tokens,test_included_special_tokens,test_filter_fallback_texts
0,cls,0.001,NaN,NaN,0.848997,0.012791,0.897802,0.860454,0.832148,0.845039,0.754617,674,36,57,143,910,910,0
1,mean,0.001,NaN,NaN,0.894202,0.013346,0.897802,0.848033,0.859085,0.853353,0.772616,659,51,42,158,15911,1820,0
2,mean_lt_0.5,0.001,NaN,0.5,0.862045,0.020454,0.887912,0.834841,0.841972,0.838316,0.748768,656,54,48,152,5709,866,2
3,mean_lt_1,0.001,NaN,1.0,0.887673,0.010071,0.893407,0.844938,0.843697,0.844315,0.756892,662,48,49,151,10328,1417,0
4,mean_lt_1.5,0.001,NaN,1.5,0.892535,0.013750,0.902198,0.865188,0.842148,0.852845,0.767624,674,36,53,147,13706,1725,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,mean_gt_0.1_lt_3.5,0.001,0.1,3.5,0.891607,0.012950,0.901099,0.859107,0.848627,0.853691,0.770408,669,41,49,151,14723,1631,0
57,mean_gt_0.1_lt_4,0.001,0.1,4.0,0.891819,0.012795,0.901099,0.859107,0.848627,0.853691,0.770408,669,41,49,151,14724,1631,0
58,mean_gt_0.1_lt_4.5,0.001,0.1,4.5,0.891819,0.012795,0.901099,0.859107,0.848627,0.853691,0.770408,669,41,49,151,14724,1631,0
59,mean_gt_0.1_lt_5,0.001,0.1,5.0,0.891819,0.012795,0.901099,0.859107,0.848627,0.853691,0.770408,669,41,49,151,14724,1631,0
